# LeetCode 60 — Permutation Sequence (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def getPermutation(self, n: int, k: int) -> str:

```

**Mục tiêu học:**

1. Hiểu **hoán vị** (permutation), **giai thừa** (factorial) và **thứ tự từ điển** (lexicographic order) là gì.
2. Biết sinh mọi hoán vị bằng **backtracking** (quay lui) và biết **dừng sớm**.
3. Biết dùng `itertools.permutations` + `itertools.islice` của thư viện chuẩn.
4. Hiểu và tự viết được thuật toán **next permutation** (hoán vị kế tiếp).
5. Nắm cách chính: **hệ đếm giai thừa** (factorial number system) — "nhảy cóc" thẳng tới hoán vị thứ `k` trong `O(n²)`, không cần liệt kê.
6. Tránh lỗi kinh điển nhất của bài: **off-by-one giữa `k` và `k - 1`**.

## 1. Đề bài

Tập hợp `[1, 2, 3, ..., n]` có tổng cộng `n!` hoán vị khác nhau.

Nếu liệt kê và đánh số tất cả các hoán vị **theo thứ tự**, với `n = 3` ta được dãy:

| Thứ tự `k` | Hoán vị |
|---:|:---:|
| 1 | `"123"` |
| 2 | `"132"` |
| 3 | `"213"` |
| 4 | `"231"` |
| 5 | `"312"` |
| 6 | `"321"` |

Cho `n` và `k`, hãy trả về **hoán vị thứ `k`** (dưới dạng chuỗi).

**Ví dụ:**

| Input | Output |
|---|---|
| `n = 3, k = 3` | `"213"` |
| `n = 4, k = 9` | `"2314"` |
| `n = 3, k = 1` | `"123"` |

**Constraints:**

- `1 <= n <= 9`
- `1 <= k <= n!`

Đừng quên hai điều nhỏ nhưng quan trọng:

- `k` được **đếm từ 1** (hoán vị đầu tiên là `k = 1`), trong khi Python đánh chỉ số từ 0.
- Kết quả là **chuỗi** `str`, không phải list số.

### Thứ tự từ điển (lexicographic order) là gì?

"Thứ tự" trong đề là **thứ tự từ điển**: giống cách sắp xếp từ trong cuốn từ điển.

So sánh hai chuỗi cùng độ dài: nhìn từ trái sang phải, tìm **vị trí đầu tiên khác nhau**, chuỗi nào có ký tự nhỏ hơn ở đó thì đứng trước.

- `"132"` vs `"213"`: khác ngay vị trí 0, `'1' < '2'` → `"132"` đứng trước.
- `"231"` vs `"213"`: giống `'2'`, khác ở vị trí 1, `'1' < '3'` → `"213"` đứng trước.

Python so sánh chuỗi đúng theo kiểu này, nên `sorted(...)` cho đúng thứ tự của đề.

In [1]:
print("132" < "213")   # True: khác ở vị trí 0, '1' < '2'
print("231" < "213")   # False: khác ở vị trí 1, '3' > '1'

words = ["321", "123", "231", "213", "132", "312"]
print(sorted(words))   # đúng dãy của đề với n = 3

True
False
['123', '132', '213', '231', '312', '321']


## 2. Kiến thức nền 1 — Hoán vị và giai thừa

**Hoán vị** của một tập là một cách **sắp xếp lại thứ tự** tất cả phần tử, mỗi phần tử dùng đúng 1 lần.

Đếm số hoán vị của `n` phần tử:

- Vị trí đầu có `n` lựa chọn.
- Vị trí thứ hai còn `n - 1` lựa chọn (đã dùng mất 1 số).
- ...
- Vị trí cuối còn đúng `1` lựa chọn.

Nhân lại: `n × (n-1) × ... × 2 × 1 = n!` (đọc là "n giai thừa").

Ví dụ: `3! = 3 × 2 × 1 = 6` — khớp với 6 hoán vị ở bảng trên. Quy ước: `0! = 1`.

Trong Python có sẵn `math.factorial(n)`.

In [2]:
import math

for n in range(0, 10):
    print(f"{n}! = {math.factorial(n)}")

0! = 1
1! = 1
2! = 2
3! = 6
4! = 24
5! = 120
6! = 720
7! = 5040
8! = 40320
9! = 362880


## 3. Kiến thức nền 2 — Backtracking (quay lui)

Nhận xét từ bảng trên: với `n = 9` có tới `362880` hoán vị. Không quá lớn với máy tính, nhưng tạo hết chỉ để lấy **một** cái thì rất phí. Ta sẽ đi từ cách "liệt kê hết" (dễ nghĩ) tới cách "tính thẳng" (tối ưu). Trước hết, học cách liệt kê.

**Backtracking** là kỹ thuật thử từng lựa chọn một, theo nhịp 3 bước:

1. **Chọn**: đặt một số chưa dùng vào vị trí tiếp theo.
2. **Đi sâu**: gọi đệ quy (recursion — hàm tự gọi lại chính nó) để điền vị trí kế tiếp.
3. **Bỏ chọn**: gỡ số vừa đặt ra để thử số khác.

Khi đã điền đủ `n` vị trí, ta có một hoán vị hoàn chỉnh.

Hai biến trạng thái cần nhớ:

- `path`: list các số đã đặt (hoán vị đang xây dở).
- `used[x]`: `True` nếu số `x` đã nằm trong `path`.

Mini ví dụ: in toàn bộ quá trình chọn / bỏ chọn với `[1, 2, 3]`. Thụt lề thể hiện độ sâu đệ quy.

In [3]:
def trace_backtrack(n):
    path = []
    used = [False] * (n + 1)          # dùng chỉ số 1..n cho dễ đọc

    def backtrack():
        indent = "  " * len(path)
        if len(path) == n:
            print(indent + "=> ghi nhận", path)
            return
        for x in range(1, n + 1):     # thử các số theo thứ tự TĂNG DẦN
            if not used[x]:
                print(indent + f"chọn {x}")
                used[x] = True
                path.append(x)        # 1. chọn
                backtrack()           # 2. đi sâu
                path.pop()            # 3. bỏ chọn
                used[x] = False

    backtrack()

trace_backtrack(3)

chọn 1
  chọn 2
    chọn 3
      => ghi nhận [1, 2, 3]
  chọn 3
    chọn 2
      => ghi nhận [1, 3, 2]
chọn 2
  chọn 1
    chọn 3
      => ghi nhận [2, 1, 3]
  chọn 3
    chọn 1
      => ghi nhận [2, 3, 1]
chọn 3
  chọn 1
    chọn 2
      => ghi nhận [3, 1, 2]
  chọn 2
    chọn 1
      => ghi nhận [3, 2, 1]


### Vì sao backtracking sinh ra đúng thứ tự từ điển?

Vì ở **mỗi vị trí** ta thử các số theo thứ tự **tăng dần** `1, 2, 3, ...`:

- Mọi hoán vị bắt đầu bằng `1` được sinh xong **trước** mọi hoán vị bắt đầu bằng `2`.
- Bên trong nhóm bắt đầu bằng `1`, nhóm `12...` lại xong trước nhóm `13...`.

Đó chính là định nghĩa thứ tự từ điển: so vị trí đầu, rồi mới tới vị trí sau.

Quan sát output trên: `[1,2,3]`, `[1,3,2]`, `[2,1,3]`, `[2,3,1]`, `[3,1,2]`, `[3,2,1]` — đúng bảng của đề.

### Mảnh ghép nhỏ: đổi list số thành chuỗi

Đề yêu cầu trả về chuỗi. `"".join(...)` chỉ nối được các **chuỗi**, nên phải đổi số sang chuỗi trước bằng `map(str, ...)`.

In [4]:
path = [2, 3, 1, 4]
print("".join(map(str, path)))      # '2314'

try:
    "".join(path)                     # lỗi: phần tử là int, không phải str
except TypeError as e:
    print("TypeError:", e)

2314
TypeError: sequence item 0: expected str instance, int found


## 4. Cách 1 — Backtracking sinh tất cả hoán vị rồi lấy thứ `k`

Ý tưởng trực giác nhất:

1. Sinh toàn bộ `n!` hoán vị theo thứ tự từ điển (bằng backtracking ở trên).
2. Trả về phần tử thứ `k`.

Chú ý: list Python đánh chỉ số từ 0, nên hoán vị thứ `k` (đếm từ 1) nằm ở `all_perms[k - 1]`.

In [5]:
def perm_backtrack_all(n: int, k: int) -> str:
    all_perms = []
    path = []
    used = [False] * (n + 1)

    def backtrack():
        if len(path) == n:
            all_perms.append("".join(map(str, path)))
            return
        for x in range(1, n + 1):
            if not used[x]:
                used[x] = True
                path.append(x)
                backtrack()
                path.pop()
                used[x] = False

    backtrack()
    return all_perms[k - 1]          # k đếm từ 1 -> chỉ số k - 1

print(perm_backtrack_all(3, 3))   # 213
print(perm_backtrack_all(4, 9))   # 2314
print(perm_backtrack_all(3, 1))   # 123

213
2314
123


### Cách 1 vướng ở đâu?

- Luôn sinh **đủ `n!`** hoán vị, kể cả khi `k = 1` (đáp án là hoán vị đầu tiên!).
- Lưu cả `n!` chuỗi vào bộ nhớ: với `n = 9` là `362880` chuỗi.

**Cải tiến nhỏ:** đếm số hoán vị đã sinh, **dừng ngay** khi đếm tới `k`, không lưu list.

Để hàm con sửa được biến đếm của hàm ngoài, ta dùng từ khóa `nonlocal` (nói với Python: "biến `count` này là của hàm bao bên ngoài, đừng tạo biến mới"). Mini ví dụ ngay dưới.

**Cách 1b — dừng sớm:** hàm `backtrack()` trả về `True` khi đã tìm thấy đáp án; mọi tầng phía trên thấy `True` thì `return True` luôn, không thử tiếp. Biến `count` đếm số hoán vị hoàn chỉnh đã đi qua, giúp ta thấy hiệu quả của việc dừng sớm.

In [6]:
def demo_nonlocal():
    count = 0
    def inc():
        nonlocal count     # thiếu dòng này sẽ báo UnboundLocalError
        count += 1
    inc(); inc(); inc()
    return count

print(demo_nonlocal())   # 3

3


In [7]:
def perm_backtrack_early(n: int, k: int, return_stats: bool = False):
    path = []
    used = [False] * (n + 1)
    count = 0
    answer = ""

    def backtrack() -> bool:
        nonlocal count, answer
        if len(path) == n:
            count += 1
            if count == k:                       # đúng hoán vị thứ k
                answer = "".join(map(str, path))
                return True                      # báo "tìm thấy rồi"
            return False
        for x in range(1, n + 1):
            if not used[x]:
                used[x] = True
                path.append(x)
                if backtrack():                  # tầng dưới tìm thấy -> dừng luôn
                    return True
                path.pop()
                used[x] = False
        return False

    backtrack()
    return (answer, count) if return_stats else answer

for n, k in [(3, 3), (4, 9), (3, 1), (9, 1), (9, 362880)]:
    ans, leaves = perm_backtrack_early(n, k, return_stats=True)
    print(f"n={n}, k={k:>6} -> {ans:<9} | số hoán vị đã đi qua: {leaves}")

n=3, k=     3 -> 213       | số hoán vị đã đi qua: 3
n=4, k=     9 -> 2314      | số hoán vị đã đi qua: 9
n=3, k=     1 -> 123       | số hoán vị đã đi qua: 1
n=9, k=     1 -> 123456789 | số hoán vị đã đi qua: 1


n=9, k=362880 -> 987654321 | số hoán vị đã đi qua: 362880


### Độ phức tạp của Cách 1 / 1b

Gọi `n` là số chữ số, `k` là thứ tự cần tìm.

| Phiên bản | Thời gian | Bộ nhớ |
|---|---|---|
| 1 — sinh hết | `O(n · n!)` (mỗi hoán vị tốn `O(n)` để tạo chuỗi, lại còn chi phí duyệt cây) | `O(n · n!)` để lưu list |
| 1b — dừng sớm | `O(n · k)` cho `k` hoán vị hoàn chỉnh (mỗi cái `O(n)`), cộng chi phí duyệt các nút trung gian của cây | `O(n)` cho `path`, `used`, đệ quy |

Output trên cho thấy: `k = 1` chỉ đi qua 1 hoán vị, nhưng `k = 9! = 362880` vẫn phải đi qua **toàn bộ** 362880 hoán vị. Trường hợp xấu nhất chưa hề cải thiện.

## 5. Cách 2 — `itertools.permutations` + `islice`

Thư viện chuẩn có sẵn công cụ làm đúng việc của Cách 1b, viết bằng C nên nhanh hơn.

**`itertools.permutations(iterable)`** trả về một **iterator** (bộ lặp: sinh từng phần tử khi được hỏi, không tạo sẵn cả list) gồm các tuple hoán vị. Tài liệu Python đảm bảo: nếu input đã **sắp xếp tăng dần** thì các hoán vị ra theo **thứ tự từ điển**.

**`itertools.islice(iterator, start, stop)`** giống slicing `a[start:stop]` nhưng dùng cho iterator: bỏ qua `start` phần tử đầu, lấy tới `stop`.

**`next(iterator)`** lấy phần tử kế tiếp của iterator.

Hoán vị thứ `k` (đếm từ 1) có chỉ số `k - 1` (đếm từ 0). Vậy ta bỏ qua `k - 1` hoán vị đầu rồi lấy cái kế tiếp:

```python
next(islice(permutations(digits), k - 1, None))
```

`stop = None` nghĩa là "không giới hạn" — nhưng ta chỉ gọi `next` một lần nên chỉ lấy đúng 1 phần tử.

In [8]:
from itertools import permutations, islice

print(list(permutations([1, 2, 3])))          # 6 tuple, đúng thứ tự từ điển

it = permutations([1, 2, 3])
print(next(it))                               # (1, 2, 3)
print(next(it))                               # (1, 3, 2)

print(list(islice(range(100), 3, 6)))         # [3, 4, 5] -- như range(100)[3:6]

[(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
(1, 2, 3)
(1, 3, 2)
[3, 4, 5]


In [9]:
from itertools import permutations, islice

def perm_itertools(n: int, k: int) -> str:
    digits = "".join(str(d) for d in range(1, n + 1))    # "123...n" (đã tăng dần)
    kth = next(islice(permutations(digits), k - 1, None))  # tuple ký tự
    return "".join(kth)

print(perm_itertools(3, 3))   # 213
print(perm_itertools(4, 9))   # 2314
print(perm_itertools(3, 1))   # 123

213
2314
123


### Độ phức tạp của Cách 2

- Thời gian: `O(n · k)` — iterator vẫn phải sinh lần lượt `k` hoán vị, mỗi cái là một tuple `n` phần tử. Worst case `k = n!`.
- Bộ nhớ: `O(n)` — iterator chỉ giữ hoán vị hiện tại.

Ngắn gọn, nhanh hơn Cách 1b nhờ chạy bằng C, nhưng **bản chất vẫn là đi từng bước một**. Và trong phỏng vấn, người ta muốn thấy bạn hiểu cơ chế bên trong.

## 6. Cách 3 — Gọi "next permutation" `k - 1` lần

### Ý tưởng

Bắt đầu từ hoán vị nhỏ nhất `[1, 2, ..., n]` (chính là hoán vị thứ 1). Nếu có hàm biến một hoán vị thành **hoán vị kế tiếp** theo thứ tự từ điển, ta gọi nó `k - 1` lần là tới hoán vị thứ `k`.

Không đệ quy, không lưu list, chỉ biến đổi **tại chỗ** (in-place) một mảng `n` phần tử.

### Thuật toán next permutation (LeetCode #31)

Quan sát: một **đuôi giảm dần** (ví dụ `...4 2` trong `1 3 4 2`) đã là cách xếp **lớn nhất** của các số trong đuôi đó, không thể tăng thêm bằng cách đảo trong đuôi. Muốn tăng, phải đụng tới số đứng ngay trước đuôi.

1. **Tìm pivot**: từ phải sang trái, tìm `i` lớn nhất sao cho `a[i] < a[i+1]`. (Mọi thứ bên phải `i` là đuôi giảm dần.) Nếu không có `i` → mảng đang là hoán vị lớn nhất.
2. **Tìm số thay thế**: từ phải sang trái, tìm `j` đầu tiên có `a[j] > a[i]` — đây là số **nhỏ nhất trong đuôi mà vẫn lớn hơn** `a[i]`.
3. **Hoán đổi** `a[i]` và `a[j]`. Đuôi vẫn giảm dần.
4. **Đảo ngược đuôi** `a[i+1:]` để nó tăng dần — cách xếp **nhỏ nhất** của đuôi.

Ví dụ `1 3 4 2`:

| Bước | Thao tác | Mảng |
|---|---|---|
| 1 | `a[2]=4 >= a[3]=2` → lùi; `a[1]=3 < a[2]=4` → pivot `i = 1` (số `3`) | `1 [3] 4 2` |
| 2 | Từ phải: `a[3]=2 <= 3` lùi; `a[2]=4 > 3` → `j = 2` | `1 [3] [4] 2` |
| 3 | Hoán đổi `a[1]`, `a[2]` | `1 4 3 2` |
| 4 | Đảo ngược đuôi `a[2:] = [3, 2]` → `[2, 3]` | `1 4 2 3` |

Kiểm tra: sau `1342` trong thứ tự từ điển đúng là `1423`.

Hai cú pháp Python sẽ dùng:

- Hoán đổi hai phần tử: `a[i], a[j] = a[j], a[i]`.
- Đảo ngược một đoạn tại chỗ: `a[i+1:] = reversed(a[i+1:])` (gán lại slice bằng bản đảo ngược).

In [10]:
a = [1, 3, 4, 2]
a[1], a[2] = a[2], a[1]
print(a)                    # [1, 4, 3, 2]
a[2:] = reversed(a[2:])
print(a)                    # [1, 4, 2, 3]

[1, 4, 3, 2]
[1, 4, 2, 3]


In [11]:
def next_permutation(a: list) -> bool:
    """Biến a thành hoán vị kế tiếp (tại chỗ). Trả về False nếu a đã là lớn nhất."""
    i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]:     # bước 1: tìm pivot
        i -= 1
    if i < 0:                               # cả mảng giảm dần: hết hoán vị
        a.reverse()
        return False
    j = len(a) - 1
    while a[j] <= a[i]:                     # bước 2: số nhỏ nhất > a[i] trong đuôi
        j -= 1
    a[i], a[j] = a[j], a[i]                 # bước 3: hoán đổi
    a[i + 1:] = reversed(a[i + 1:])         # bước 4: đảo đuôi thành tăng dần
    return True

a = [1, 2, 3]
print(a)
while next_permutation(a):
    print(a)

[1, 2, 3]
[1, 3, 2]
[2, 1, 3]
[2, 3, 1]
[3, 1, 2]
[3, 2, 1]


### Trace next permutation

Hàm `trace_next_permutation` in pivot `i`, số thay thế `j`, mảng sau khi hoán đổi và sau khi đảo đuôi. Ta thử `1 3 4 2` (khớp bảng trên) và `2 4 3 1` (đuôi dài 3 phần tử).

In [12]:
def trace_next_permutation(a: list) -> None:
    a = a[:]                                # chép để không sửa list gốc
    print("Ban đầu:", a)
    i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]:
        i -= 1
    if i < 0:
        print("  Không có pivot -> đã là hoán vị lớn nhất")
        return
    print(f"  pivot i={i}, a[i]={a[i]}, đuôi giảm dần = {a[i + 1:]}")
    j = len(a) - 1
    while a[j] <= a[i]:
        j -= 1
    print(f"  số thay thế j={j}, a[j]={a[j]}")
    a[i], a[j] = a[j], a[i]
    print("  sau hoán đổi:  ", a)
    a[i + 1:] = reversed(a[i + 1:])
    print("  sau đảo đuôi:  ", a)

trace_next_permutation([1, 3, 4, 2])
print()
trace_next_permutation([2, 4, 3, 1])
print()
trace_next_permutation([3, 2, 1])

Ban đầu: [1, 3, 4, 2]
  pivot i=1, a[i]=3, đuôi giảm dần = [4, 2]
  số thay thế j=2, a[j]=4
  sau hoán đổi:   [1, 4, 3, 2]
  sau đảo đuôi:   [1, 4, 2, 3]

Ban đầu: [2, 4, 3, 1]
  pivot i=0, a[i]=2, đuôi giảm dần = [4, 3, 1]
  số thay thế j=2, a[j]=3
  sau hoán đổi:   [3, 4, 2, 1]
  sau đảo đuôi:   [3, 1, 2, 4]

Ban đầu: [3, 2, 1]
  Không có pivot -> đã là hoán vị lớn nhất


In [13]:
def perm_next_k(n: int, k: int) -> str:
    a = list(range(1, n + 1))      # hoán vị thứ 1
    for _ in range(k - 1):         # đi thêm k - 1 bước -> hoán vị thứ k
        next_permutation(a)
    return "".join(map(str, a))

print(perm_next_k(3, 3))   # 213
print(perm_next_k(4, 9))   # 2314
print(perm_next_k(3, 1))   # 123

213
2314
123


### Độ phức tạp của Cách 3

- Mỗi lần `next_permutation` tốn `O(n)` (worst case). Gọi `k - 1` lần → **Thời gian `O(n · k)`**, worst case `O(n · n!)`.
- **Bộ nhớ `O(1)`** ngoài mảng `a` (độ dài `n`) và chuỗi kết quả.

### Cả 3 cách đều vướng chung một chỗ

Chúng đều **đi từng bước một** qua `k` hoán vị. Câu hỏi then chốt:

> Có cách nào **nhảy cóc** qua cả một nhóm lớn hoán vị mà không cần liệt kê chúng?

Có! Vì ta **đếm được** chính xác mỗi nhóm có bao nhiêu hoán vị.

## 7. Bước đột phá — Chia hoán vị thành từng nhóm

Xét `n = 4`, có `4! = 24` hoán vị. Chia theo **chữ số đầu tiên**:

- Nếu chữ số đầu đã cố định, 3 chữ số còn lại xếp tùy ý → mỗi nhóm có `3! = 6` hoán vị.
- Theo thứ tự từ điển, nhóm "đầu 1" đứng trước, rồi "đầu 2", "đầu 3", "đầu 4".

| Nhóm | Chữ số đầu | `k` (đếm từ 1) | `k - 1` (đếm từ 0) |
|---:|:---:|:---:|:---:|
| 0 | `1` | 1 – 6 | 0 – 5 |
| 1 | `2` | 7 – 12 | 6 – 11 |
| 2 | `3` | 13 – 18 | 12 – 17 |
| 3 | `4` | 19 – 24 | 18 – 23 |

Với `k = 9`: `k` nằm trong khoảng 7–12 → chữ số đầu là **`2`**. Ta biết ngay điều đó mà **không cần** liệt kê 8 hoán vị trước.

Tiếp tục **bên trong nhóm "đầu 2"**: còn lại `[1, 3, 4]`, chia tiếp theo chữ số thứ hai, mỗi nhóm con có `2! = 2` hoán vị:

| Nhóm con | Bắt đầu bằng | `k` | Các hoán vị |
|---:|:---:|:---:|---|
| 0 | `21` | 7 – 8 | `2134`, `2143` |
| 1 | `23` | 9 – 10 | `2314`, `2341` |
| 2 | `24` | 11 – 12 | `2413`, `2431` |

`k = 9` rơi vào nhóm con `23`, và là hoán vị **đầu tiên** của nhóm con đó → còn `[1, 4]` xếp tăng dần → **`2314`**. Khớp Example 2!

Cell dưới in toàn bộ 24 hoán vị `n = 4`, mỗi dòng là một nhóm, để kiểm chứng bảng.

In [14]:
from itertools import permutations

all4 = ["".join(p) for p in permutations("1234")]
block = math.factorial(3)        # 6 hoán vị mỗi nhóm
for g in range(4):
    row = all4[g * block:(g + 1) * block]
    ks = f"k={g * block + 1}..{(g + 1) * block}"
    print(f"Nhóm {g} ({ks:>8}):", " ".join(row))

print("Hoán vị thứ 9:", all4[9 - 1])

Nhóm 0 (  k=1..6): 1234 1243 1324 1342 1423 1432
Nhóm 1 ( k=7..12): 2134 2143 2314 2341 2413 2431
Nhóm 2 (k=13..18): 3124 3142 3214 3241 3412 3421
Nhóm 3 (k=19..24): 4123 4132 4213 4231 4312 4321
Hoán vị thứ 9: 2314


### Công thức chọn nhóm — và cái bẫy `k` vs `k - 1`

Muốn biết `k` thuộc nhóm nào, tự nhiên ta nghĩ tới phép chia nguyên `k // 6`. Nhưng thử:

- `k = 6` (hoán vị cuối nhóm 0): `6 // 6 = 1` → **sai**, nhảy sang nhóm 1.
- `k = 12` (cuối nhóm 1): `12 // 6 = 2` → **sai**.

Lý do: chia nguyên hợp với chỉ số **đếm từ 0**. Các phần tử `0..5` thuộc nhóm 0, `6..11` thuộc nhóm 1, ... Vì vậy phải **đổi sang 0-indexed trước**:

```python
k0 = k - 1
group = k0 // 6        # đúng
```

Cell dưới so sánh hai công thức ở các vị trí biên với nhóm thật (lấy từ danh sách `all4`).

In [15]:
print(" k | k//6 | (k-1)//6 | nhóm thật")
for k in [1, 5, 6, 7, 12, 13, 18, 19, 24]:
    true_group = int(all4[k - 1][0]) - 1          # chữ số đầu 1 -> nhóm 0, ...
    wrong, right = k // 6, (k - 1) // 6
    flag = "" if wrong == true_group else "  <- k//6 SAI"
    print(f"{k:>2} | {wrong:>4} | {right:>8} | {true_group:>9}{flag}")

 k | k//6 | (k-1)//6 | nhóm thật
 1 |    0 |        0 |         0
 5 |    0 |        0 |         0
 6 |    1 |        0 |         0  <- k//6 SAI
 7 |    1 |        1 |         1
12 |    2 |        1 |         1  <- k//6 SAI
13 |    2 |        2 |         2
18 |    3 |        2 |         2  <- k//6 SAI
19 |    3 |        3 |         3
24 |    4 |        3 |         3  <- k//6 SAI


### `divmod`: lấy thương và dư cùng lúc

Sau khi biết nhóm, ta cần biết **vị trí bên trong nhóm** để làm tiếp — đó chính là **phần dư**.

`divmod(a, b)` trả về tuple `(a // b, a % b)`.

Với `k = 9`: `k0 = 8`, `divmod(8, 6) = (1, 2)` → nhóm 1 (chữ số `2`), và là phần tử số 2 (đếm từ 0) bên trong nhóm.

### `list.pop(i)`: lấy ra và xóa phần tử ở vị trí `i`

Mỗi chữ số chỉ được dùng một lần. Ta giữ list `remaining` các chữ số **chưa dùng** (luôn tăng dần); khi chọn chữ số ở vị trí `idx` thì `remaining.pop(idx)` vừa trả về chữ số đó vừa xóa nó khỏi list. Vì xóa giữa list phải dồn các phần tử phía sau, `pop(i)` tốn `O(n)`.

Cell dưới thử cả hai công cụ.

In [16]:
print(divmod(8, 6))     # (1, 2)
q, r = divmod(8, 6)
print("thương =", q, "| dư =", r)
print(divmod(5, 6))     # (0, 5): k=6 -> k0=5 vẫn ở nhóm 0, vị trí cuối
print()

remaining = ['1', '2', '3', '4']
chosen = remaining.pop(1)
print(chosen, remaining)     # 2 ['1', '3', '4']
chosen = remaining.pop(1)
print(chosen, remaining)     # 3 ['1', '4']

(1, 2)
thương = 1 | dư = 2
(0, 5)

2 ['1', '3', '4']
3 ['1', '4']


## 8. Hệ đếm giai thừa (factorial number system)

Những gì ta vừa làm có một cái tên: **hệ đếm giai thừa** (còn gọi là *factoradic*).

So sánh với hệ thập phân quen thuộc:

- Thập phân: `345 = 3·100 + 4·10 + 5·1`. Trọng số các vị trí là `100, 10, 1`; mỗi chữ số từ `0` tới `9`.
- Giai thừa (với `n = 4`): `x = d0·3! + d1·2! + d2·1! + d3·0!`. Trọng số là `6, 2, 1, 1`; chữ số `d0 ≤ 3`, `d1 ≤ 2`, `d2 ≤ 1`, `d3 = 0`.

Mọi số `x` từ `0` tới `n! - 1` viết được **duy nhất** theo cách này. Ví dụ `x = 8 = 1·6 + 1·2 + 0·1 + 0·1` → chữ số `[1, 1, 0, 0]`.

Ý nghĩa với bài toán: **chữ số thứ `i` cho biết chọn phần tử ở vị trí nào trong list `remaining`** tại bước `i`. Với `k = 9` → `x = k - 1 = 8` → `[1, 1, 0, 0]`:

- chọn `remaining[1]` của `[1,2,3,4]` → `2`
- chọn `remaining[1]` của `[1,3,4]` → `3`
- chọn `remaining[0]` của `[1,4]` → `1`
- chọn `remaining[0]` của `[4]` → `4`

→ `"2314"`.

In [17]:
def to_factoradic(x: int, n: int) -> list:
    """Viết x (0 <= x < n!) trong hệ đếm giai thừa, trả về n chữ số."""
    digits = []
    for i in range(n - 1, -1, -1):          # trọng số (n-1)!, ..., 1!, 0!
        d, x = divmod(x, math.factorial(i))
        digits.append(d)
    return digits

print(to_factoradic(8, 4))    # [1, 1, 0, 0]
print(to_factoradic(23, 4))   # [3, 2, 1, 0] -- số lớn nhất, ứng với "4321"
print(to_factoradic(0, 4))    # [0, 0, 0, 0] -- ứng với "1234"
print()

# Bảng đầy đủ cho n = 3: đếm x = 0, 1, 2, ... trong hệ giai thừa
# giống hệt việc đi qua các hoán vị theo thứ tự từ điển.
def factoradic_to_perm(fd: list, n: int) -> str:
    remaining = [str(d) for d in range(1, n + 1)]
    return "".join(remaining.pop(d) for d in fd)

print(" k | x=k-1 | factoradic | hoán vị")
for k in range(1, 7):
    fd = to_factoradic(k - 1, 3)
    print(f"{k:>2} | {k - 1:>5} | {str(fd):>10} | {factoradic_to_perm(fd, 3)}")

[1, 1, 0, 0]
[3, 2, 1, 0]
[0, 0, 0, 0]

 k | x=k-1 | factoradic | hoán vị
 1 |     0 |  [0, 0, 0] | 123
 2 |     1 |  [0, 1, 0] | 132
 3 |     2 |  [1, 0, 0] | 213
 4 |     3 |  [1, 1, 0] | 231
 5 |     4 |  [2, 0, 0] | 312
 6 |     5 |  [2, 1, 0] | 321


## 9. Cách 4 (cách chính) — Tính thẳng hoán vị thứ `k`

Ghép tất cả lại:

1. `remaining = ['1', '2', ..., 'n']` — các chữ số chưa dùng, tăng dần.
2. Tính sẵn giai thừa `fact[i] = i!` cho `i = 0..n-1`.
3. **`k -= 1`** — đổi sang đếm từ 0 (bước quan trọng nhất!).
4. Lặp `n` lần; ở mỗi bước còn `m` chữ số chưa dùng (`m = n, n-1, ..., 1`):
   - `block = fact[m - 1]` = số hoán vị trong mỗi nhóm con.
   - `idx, k = divmod(k, block)` — `idx` là nhóm, `k` là vị trí bên trong nhóm.
   - Lấy `remaining.pop(idx)` nối vào kết quả.
5. Nối thành chuỗi.

In [18]:
def perm_factorial(n: int, k: int) -> str:
    remaining = [str(d) for d in range(1, n + 1)]   # chữ số chưa dùng, tăng dần
    fact = [1] * n                                  # fact[i] = i!
    for i in range(1, n):
        fact[i] = fact[i - 1] * i

    k -= 1                                          # đếm từ 0
    result = []
    for m in range(n, 0, -1):                       # m = số chữ số còn lại
        block = fact[m - 1]                         # mỗi nhóm có (m-1)! hoán vị
        idx, k = divmod(k, block)                   # nhóm thứ idx, vị trí k trong nhóm
        result.append(remaining.pop(idx))
    return "".join(result)

print(perm_factorial(3, 3))   # 213
print(perm_factorial(4, 9))   # 2314
print(perm_factorial(3, 1))   # 123

213
2314
123


## 10. Mô phỏng từng bước — `n = 4, k = 9`

Sau `k -= 1`, ta có `k = 8`.

| Bước | `m` (còn lại) | `remaining` trước | `block = (m-1)!` | `divmod(k, block)` | `idx` | Chọn | `k` mới | Kết quả |
|---:|---:|---|---:|---|---:|:---:|---:|---|
| 1 | 4 | `[1,2,3,4]` | 6 | `divmod(8, 6) = (1, 2)` | 1 | `2` | 2 | `2` |
| 2 | 3 | `[1,3,4]` | 2 | `divmod(2, 2) = (1, 0)` | 1 | `3` | 0 | `23` |
| 3 | 2 | `[1,4]` | 1 | `divmod(0, 1) = (0, 0)` | 0 | `1` | 0 | `231` |
| 4 | 1 | `[4]` | 1 | `divmod(0, 1) = (0, 0)` | 0 | `4` | 0 | `2314` |

Ý nghĩa từng cột:

- `block`: nếu cố định chữ số ở vị trí hiện tại, các chữ số còn lại tạo ra bao nhiêu hoán vị.
- `idx`: phải **bỏ qua** bao nhiêu nhóm trọn vẹn → chọn chữ số thứ `idx` trong `remaining`.
- `k` mới: vị trí còn lại **bên trong** nhóm vừa chọn.

Bước 1 ứng với bảng nhóm ở mục 7 (nhóm 1 = "đầu 2"), bước 2 ứng với bảng nhóm con (nhóm con 1 = "23").

Cell trace dưới chạy thêm **ca biên `k = n!`** (hoán vị cuối cùng): `n = 3, k = 6`, tức `k - 1 = 5`. Đây là ca dễ "toang" nhất nếu quên trừ 1 (xem mục Lỗi thường gặp). Kết quả phải là `"321"` — mọi `idx` đều chọn phần tử **cuối** của `remaining`.

In [19]:
def trace_perm_factorial(n: int, k: int) -> str:
    remaining = [str(d) for d in range(1, n + 1)]
    fact = [1] * n
    for i in range(1, n):
        fact[i] = fact[i - 1] * i

    print(f"n={n}, k={k} -> đổi sang đếm từ 0: k={k - 1}")
    k -= 1
    result = []
    for step, m in enumerate(range(n, 0, -1), start=1):
        block = fact[m - 1]
        before = remaining[:]
        old_k = k
        idx, k = divmod(k, block)
        chosen = remaining.pop(idx)
        result.append(chosen)
        print(f"Bước {step}: remaining={before}, block={block}, "
              f"divmod({old_k}, {block})=({idx}, {k}) -> chọn '{chosen}', "
              f"kết quả='{''.join(result)}'")
    return "".join(result)

print("=>", trace_perm_factorial(4, 9))
print()
print("=>", trace_perm_factorial(3, 6))

n=4, k=9 -> đổi sang đếm từ 0: k=8
Bước 1: remaining=['1', '2', '3', '4'], block=6, divmod(8, 6)=(1, 2) -> chọn '2', kết quả='2'
Bước 2: remaining=['1', '3', '4'], block=2, divmod(2, 2)=(1, 0) -> chọn '3', kết quả='23'
Bước 3: remaining=['1', '4'], block=1, divmod(0, 1)=(0, 0) -> chọn '1', kết quả='231'
Bước 4: remaining=['4'], block=1, divmod(0, 1)=(0, 0) -> chọn '4', kết quả='2314'
=> 2314

n=3, k=6 -> đổi sang đếm từ 0: k=5
Bước 1: remaining=['1', '2', '3'], block=2, divmod(5, 2)=(2, 1) -> chọn '3', kết quả='3'
Bước 2: remaining=['1', '2'], block=1, divmod(1, 1)=(1, 0) -> chọn '2', kết quả='32'
Bước 3: remaining=['1'], block=1, divmod(0, 1)=(0, 0) -> chọn '1', kết quả='321'
=> 321


### Độ phức tạp của Cách 4

Gọi `n` là số chữ số.

- Tính bảng `fact`: `O(n)`.
- Vòng lặp chạy `n` lần; mỗi lần `divmod` là `O(1)` nhưng `remaining.pop(idx)` là `O(n)` (dồn phần tử).
- **Thời gian: `O(n²)`**. Với `n ≤ 9`, đó chỉ là vài chục thao tác — **không phụ thuộc `k`**!
- **Bộ nhớ: `O(n)`** cho `remaining`, `fact`, `result`.

(Có thể giảm xuống `O(n log n)` bằng cây Fenwick/segment tree để tìm "phần tử thứ `idx` còn lại", nhưng với `n ≤ 9` hoàn toàn không cần.)

## 11. Code nộp LeetCode

Chuyển `perm_factorial` vào đúng signature `class Solution`. Dùng `math.factorial` cho gọn (LeetCode đã import sẵn `math`, nhưng ta vẫn tự import cho an toàn khi chạy ở nơi khác).

In [20]:
import math

class Solution:
    def getPermutation(self, n: int, k: int) -> str:
        remaining = [str(d) for d in range(1, n + 1)]
        k -= 1                                    # đổi sang đếm từ 0
        result = []
        for m in range(n, 0, -1):                 # m chữ số còn lại
            idx, k = divmod(k, math.factorial(m - 1))
            result.append(remaining.pop(idx))
        return "".join(result)

### Test bằng `assert`

Gồm 3 ví dụ của đề và các ca biên:

- `n = 1` (chỉ có một hoán vị).
- `k = 1` (hoán vị nhỏ nhất) và `k = n!` (lớn nhất).
- `k` ở **biên nhóm**: `k = 6` (cuối nhóm "đầu 1") và `k = 7` (đầu nhóm "đầu 2") với `n = 4`.
- `n = 9` lớn nhất theo constraints.

In [21]:
sol = Solution()

tests = [
    (3, 3, "213"),            # Example 1
    (4, 9, "2314"),           # Example 2
    (3, 1, "123"),            # Example 3
    (1, 1, "1"),              # n nhỏ nhất
    (2, 2, "21"),
    (4, 6, "1432"),           # cuối nhóm "đầu 1"
    (4, 7, "2134"),           # đầu nhóm "đầu 2"
    (4, 24, "4321"),          # k = n!
    (9, 1, "123456789"),
    (9, 362880, "987654321"), # k = 9!
]

for n, k, expected in tests:
    got = sol.getPermutation(n, k)
    print(f"n={n}, k={k:>6} -> {got:<10} expected={expected}")
    assert got == expected

print("Tất cả test PASS!")

n=3, k=     3 -> 213        expected=213
n=4, k=     9 -> 2314       expected=2314
n=3, k=     1 -> 123        expected=123
n=1, k=     1 -> 1          expected=1
n=2, k=     2 -> 21         expected=21
n=4, k=     6 -> 1432       expected=1432
n=4, k=     7 -> 2134       expected=2134
n=4, k=    24 -> 4321       expected=4321
n=9, k=     1 -> 123456789  expected=123456789
n=9, k=362880 -> 987654321  expected=987654321
Tất cả test PASS!


### Đối chiếu chéo các cách

Hai tầng kiểm tra:

1. **Vét cạn** mọi `(n, k)` với `n ≤ 6` (tổng `1 + 2 + 6 + 24 + 120 + 720 = 873` cặp): mọi cách phải khớp nhau.
2. **Ngẫu nhiên** 200 cặp với `n ≤ 8`, dùng `random.seed` cố định để kết quả lặp lại được.

Ở tầng 1, để nhanh, ta sinh danh sách chuẩn một lần cho mỗi `n` thay vì gọi `perm_backtrack_all` cho từng `k`.

In [22]:
import random

methods = {
    "backtrack_early": perm_backtrack_early,
    "itertools": perm_itertools,
    "next_k": perm_next_k,
    "factorial": perm_factorial,
    "Solution": lambda n, k: Solution().getPermutation(n, k),
}

# Tầng 1: vét cạn n <= 6, so với danh sách chuẩn sinh bằng sorted()
checked = 0
for n in range(1, 7):
    reference = sorted("".join(p) for p in permutations("123456789"[:n]))
    for k in range(1, math.factorial(n) + 1):
        for name, f in methods.items():
            assert f(n, k) == reference[k - 1], (name, n, k)
        checked += 1
print("Vét cạn: khớp", checked, "cặp (n, k)")

# Tầng 2: ngẫu nhiên n <= 8, kèm cả perm_backtrack_all
random.seed(60)
for _ in range(200):
    n = random.randint(1, 8)
    k = random.randint(1, math.factorial(n))
    expected = perm_backtrack_all(n, k)
    for name, f in methods.items():
        assert f(n, k) == expected, (name, n, k)
print("Ngẫu nhiên: 200 cặp đều khớp giữa", len(methods) + 1, "cách")

Vét cạn: khớp 873 cặp (n, k)


Ngẫu nhiên: 200 cặp đều khớp giữa 6 cách


### Cảm nhận tốc độ với worst case `n = 9, k = 9!`

Con số cụ thể tùy máy, nhưng thứ bậc rất rõ: các cách "đi từng bước" tốn hàng trăm nghìn bước, còn cách giai thừa chỉ vài phép tính.

In [23]:
import time

n, k = 9, math.factorial(9)
for name, f in [("backtrack_all", perm_backtrack_all), *methods.items()]:
    start = time.perf_counter()
    ans = f(n, k)
    ms = (time.perf_counter() - start) * 1000
    print(f"{name:<16} -> {ans}  ({ms:9.3f} ms)")

backtrack_all    -> 987654321  (  451.111 ms)


backtrack_early  -> 987654321  (  270.014 ms)
itertools        -> 987654321  (    3.562 ms)
next_k           -> 987654321  (  140.546 ms)
factorial        -> 987654321  (    0.026 ms)
Solution         -> 987654321  (    0.012 ms)


## 12. Vì sao Cách 4 đúng?

**Invariant (bất biến) ở đầu mỗi vòng lặp**, khi còn `m` chữ số trong `remaining`:

> Đáp án = `result` (phần đã chốt) + **hoán vị thứ `k` (đếm từ 0)** của các chữ số trong `remaining`, theo thứ tự từ điển.

- **Ban đầu**: `result` rỗng, `remaining = [1..n]`, `k = k_gốc - 1` → đúng nguyên đề bài (chỉ đổi sang đếm từ 0).
- **Mỗi vòng**: các hoán vị của `remaining` chia thành `m` nhóm liên tiếp theo chữ số đầu, mỗi nhóm đúng `(m-1)!` phần tử, nhóm nhỏ trước (vì `remaining` tăng dần). Vậy hoán vị thứ `k` nằm ở nhóm `idx = k // (m-1)!` và là phần tử thứ `k % (m-1)!` trong nhóm đó. Chốt chữ số `remaining[idx]`, bỏ nó ra, cập nhật `k` → invariant vẫn đúng với `m - 1` chữ số.
- **Kết thúc**: `remaining` rỗng, `result` là đáp án.

**Vì sao `idx` không bao giờ vượt quá `m - 1`?** Vì invariant đảm bảo `0 ≤ k < m!`, nên `k // (m-1)! < m!/(m-1)! = m`. Điều này chỉ đúng khi ta đã trừ 1 — nếu dùng `k` đếm từ 1, `k` có thể bằng `m!` và `idx = m` → `IndexError`.

**Vì sao `remaining` luôn tăng dần?** Ban đầu tăng dần, và `pop` một phần tử không phá thứ tự các phần tử còn lại.

## 13. So sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Nhận xét |
|---|---|---|---|---|
| 1. Backtracking sinh hết | Sinh `n!` hoán vị, lấy `[k-1]` | `O(n · n!)` | `O(n · n!)` | Dễ nghĩ nhất, phí nhất |
| 1b. Backtracking dừng sớm | Đếm tới `k` thì dừng | `O(n · k)` | `O(n)` | Tốt khi `k` nhỏ, worst case như cũ |
| 2. `permutations` + `islice` | Iterator của thư viện chuẩn | `O(n · k)` | `O(n)` | Code 2 dòng, nhanh nhờ C, nhưng không "dạy" gì |
| 3. Next permutation `k-1` lần | Biến đổi tại chỗ từng bước | `O(n · k)` | `O(1)` thêm | Thuật toán quý, dùng lại ở #31 |
| **4. Hệ đếm giai thừa** | Nhảy cóc theo nhóm `(m-1)!` | **`O(n²)`** | `O(n)` | **Cách chính**, không phụ thuộc `k` |

Với constraints `n ≤ 9`, các cách 1b/2/3 vẫn có thể được LeetCode chấp nhận nhưng chậm (đặc biệt Python). Cách 4 là lời giải mong đợi ở bài Hard này.

## 14. Các lỗi người mới hay gặp

### Lỗi 1 — Quên `k -= 1` (off-by-one kinh điển)

Code sai dùng thẳng `k` (đếm từ 1) để chia:

In [24]:
def perm_bug_no_minus_one(n: int, k: int) -> str:
    remaining = [str(d) for d in range(1, n + 1)]
    result = []
    for m in range(n, 0, -1):
        idx, k = divmod(k, math.factorial(m - 1))   # SAI: k chưa đổi sang đếm từ 0
        result.append(remaining.pop(idx))
    return "".join(result)

print("n=3, k=3:", perm_bug_no_minus_one(3, 3), "| đúng là", perm_factorial(3, 3))
try:
    print(perm_bug_no_minus_one(3, 6))
except IndexError as e:
    print("n=3, k=6: IndexError:", e)

n=3, k=3: 231 | đúng là 213
n=3, k=6: IndexError: pop index out of range


**Vì sao sai?** `divmod` chia đúng nhóm khi chỉ số **đếm từ 0** (mục 7). Dùng `k` đếm từ 1 làm lệch nhóm (`k` ở cuối nhóm bị đẩy sang nhóm sau) và lệch luôn phần dư truyền xuống các bước sau — vì thế `n=3, k=3` ra `231` thay vì `213`. Còn `k = n!` cho `idx = n`, vượt chỉ số list.

**Sửa:** thêm `k -= 1` **một lần duy nhất** trước vòng lặp. (Đừng trừ 1 lại trong mỗi vòng — phần dư `k % block` vốn đã là chỉ số đếm từ 0.)

### Lỗi 2 — Quên cập nhật `k` thành phần dư

Chỉ tính `idx = k // block` mà không gán `k = k % block`, nên các bước sau vẫn dùng `k` của bước đầu.

In [25]:
def perm_bug_no_remainder(n: int, k: int) -> str:
    remaining = [str(d) for d in range(1, n + 1)]
    k -= 1
    result = []
    for m in range(n, 0, -1):
        idx = k // math.factorial(m - 1)           # SAI: không cập nhật k
        result.append(remaining.pop(idx))
    return "".join(result)

print("n=4, k=1:", perm_bug_no_remainder(4, 1), "| đúng là", perm_factorial(4, 1))
for k in [2, 9]:
    try:
        print(f"n=4, k={k}:", perm_bug_no_remainder(4, k))
    except IndexError as e:
        print(f"n=4, k={k}: IndexError:", e)

n=4, k=1: 1234 | đúng là 1234
n=4, k=2: IndexError: pop index out of range
n=4, k=9: IndexError: pop index out of range


**Vì sao sai?** Sau khi chọn nhóm, ta chỉ còn quan tâm vị trí **bên trong** nhóm đó. Bỏ quên phần dư thì với `k = 9`, bước 2 lại chia `k = 8` cho `2! = 2`, được `idx = 4` vượt list chỉ còn 3 phần tử. Nguy hiểm ở chỗ: test `k = 1` vẫn PASS (vì `k - 1 = 0`, chia gì cũng ra 0), dễ khiến ta tưởng code đúng. Vì vậy luôn test cả ca `k` lớn.

**Sửa:** `idx, k = divmod(k, block)`.

### Lỗi 3 — Không xóa chữ số đã dùng

Dùng `remaining[idx]` thay cho `remaining.pop(idx)`:

In [26]:
def perm_bug_no_pop(n: int, k: int) -> str:
    remaining = [str(d) for d in range(1, n + 1)]
    k -= 1
    result = []
    for m in range(n, 0, -1):
        idx, k = divmod(k, math.factorial(m - 1))
        result.append(remaining[idx])               # SAI: không xóa khỏi remaining
    return "".join(result)

print("n=4, k=9:", perm_bug_no_pop(4, 9), "| đúng là", perm_factorial(4, 9))

n=4, k=9: 2211 | đúng là 2314


**Vì sao sai?** Kết quả `2211` có chữ số trùng (hai chữ số `2`, hai chữ số `1`) — không còn là hoán vị. Hơn nữa, `idx` là chỉ số trong danh sách **các số chưa dùng**, nên list phải co lại sau mỗi lần chọn.

### Các lỗi khác cần nhớ

| Lỗi | Ví dụ sai | Sửa |
|---|---|---|
| Dùng sai kích thước nhóm | `block = math.factorial(m)` | Mỗi nhóm là `(m - 1)!` vì đã cố định 1 chữ số |
| Trả về list hoặc nối số nguyên | `return result` / `"".join([2, 3, 1, 4])` | Lưu chữ số dạng `str`, `return "".join(result)` |
| Cách 2 quên trừ 1 trong `islice` | `islice(perms, k, None)` | `islice(perms, k - 1, None)` |
| Cách 3 gọi next permutation `k` lần | `for _ in range(k)` | Hoán vị ban đầu đã là thứ 1, chỉ cần `k - 1` lần |
| `remaining` không tăng dần | `remaining = ['3', '1', '2']` | Phải bắt đầu tăng dần thì nhóm mới xếp đúng thứ tự từ điển |

## 15. Mẹo nhận dạng pattern

- Đề hỏi **"phần tử thứ `k`"** trong một danh sách **khổng lồ nhưng có cấu trúc** (hoán vị, tổ hợp, chuỗi theo thứ tự từ điển) → đừng liệt kê; hãy **đếm số phần tử mỗi nhóm** rồi trừ/chia dần để nhảy cóc.
- Thấy `n!` trong constraints (`1 <= k <= n!`) → nghĩ ngay tới **hệ đếm giai thừa**.
- Khi đếm thứ tự mà đề đánh số từ 1 → **đổi sang 0-indexed ngay dòng đầu**, rồi mới dùng `//`, `%`.
- Câu hỏi "hoán vị **kế tiếp**" → thuật toán next permutation (pivot → số thay thế → swap → reverse).
- Câu hỏi ngược "hoán vị này đứng **thứ mấy**" → cũng dùng hệ giai thừa, nhưng chiều ngược lại (xem Bài 5).

## 16. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 31 — Next Permutation | Chính là thuật toán của Cách 3 |
| LeetCode 46 — Permutations | Sinh mọi hoán vị bằng backtracking (Cách 1) |
| LeetCode 47 — Permutations II | Hoán vị khi có phần tử trùng — đếm nhóm phải chia cho giai thừa của số lần lặp (xem Bài 6) |
| Bài "rank của hoán vị" | Ngược của bài này: cho hoán vị, tìm `k` (Bài 5) |

## 17. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem **Gợi ý** (mục 18) và **Đáp án** (mục 19, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

```python
remaining = ['1', '2', '3', '4']
k = 13 - 1
idx, k = divmod(k, 6)
chosen = remaining.pop(idx)
print(idx, k, chosen, remaining)
```

Dòng `print` in ra gì? Chữ số đầu tiên của hoán vị thứ 13 với `n = 4` là gì?

### Bài 2 — Tính tay

Lập bảng trace giống mục 10 cho `n = 4, k = 17`. Kết quả là chuỗi nào? Sau đó viết `k - 1 = 16` trong hệ đếm giai thừa (4 chữ số).

### Bài 3 — Điền TODO cho cách chính

Hoàn thiện `exercise_get_permutation(n, k)` theo Cách 4.

In [27]:
def exercise_get_permutation(n: int, k: int) -> str:
    # TODO 1: tạo remaining = ['1', ..., 'n']
    # TODO 2: đổi k sang đếm từ 0
    # TODO 3: lặp m = n, n-1, ..., 1:
    #           block = (m-1)!
    #           idx, k = divmod(k, block)
    #           thêm remaining.pop(idx) vào kết quả
    # TODO 4: trả về chuỗi
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_get_permutation(3, 3) == "213"
# assert exercise_get_permutation(4, 9) == "2314"
# assert exercise_get_permutation(4, 24) == "4321"

### Bài 4 — Tự viết lại next permutation

Viết `exercise_next_permutation(a)` biến đổi **tại chỗ** list `a` thành hoán vị kế tiếp, trả về `False` nếu `a` đã là hoán vị lớn nhất (khi đó đưa `a` về hoán vị nhỏ nhất). Không nhìn lại mục 6.

In [28]:
def exercise_next_permutation(a: list) -> bool:
    # TODO: tìm pivot i, tìm j, hoán đổi, đảo đuôi
    raise NotImplementedError

# Gợi ý test:
# a = [1, 3, 4, 2]; assert exercise_next_permutation(a) and a == [1, 4, 2, 3]
# a = [3, 2, 1]; assert not exercise_next_permutation(a) and a == [1, 2, 3]

### Bài 5 — Biến thể: hoán vị này đứng thứ mấy?

Bài ngược lại: cho chuỗi `perm` là một hoán vị của `"12...n"`, trả về `k` (đếm từ 1) của nó.

```python
permutation_rank("2314") -> 9
permutation_rank("123")  -> 1
permutation_rank("321")  -> 6
```

In [29]:
def exercise_permutation_rank(perm: str) -> int:
    # TODO: với mỗi ký tự, đếm có bao nhiêu ký tự CÒN LẠI nhỏ hơn nó,
    #       nhân với giai thừa thích hợp, cộng dồn; nhớ +1 ở cuối
    pass

# Gợi ý test:
# assert exercise_permutation_rank("2314") == 9
# assert exercise_permutation_rank("123") == 1
# assert exercise_permutation_rank("321") == 6

### Bài 6 — Mini challenge: hoán vị thứ `k` khi có ký tự trùng

Cho chuỗi `s` (có thể có ký tự lặp, ví dụ `"aabc"`) và `k` (đếm từ 1). Xét các hoán vị **khác nhau** của `s` theo thứ tự từ điển, trả về hoán vị thứ `k`.

`"aabc"` có `4! / 2! = 12` hoán vị khác nhau: `aabc, aacb, abac, abca, acab, acba, baac, baca, bcaa, caab, caba, cbaa`.

```python
kth_unique_permutation("aabc", 1)  -> "aabc"
kth_unique_permutation("aabc", 7)  -> "baac"
kth_unique_permutation("aabc", 12) -> "cbaa"
```

Yêu cầu: **không** liệt kê hết các hoán vị.

In [30]:
def exercise_kth_unique_permutation(s: str, k: int) -> str:
    # TODO: ở mỗi vị trí, thử từng ký tự KHÁC NHAU còn lại theo thứ tự tăng dần;
    #       đếm số hoán vị khác nhau của phần còn lại nếu đặt ký tự đó;
    #       nếu k <= số đó thì chọn ký tự, ngược lại k -= số đó
    pass

# Gợi ý test:
# assert exercise_kth_unique_permutation("aabc", 1) == "aabc"
# assert exercise_kth_unique_permutation("aabc", 7) == "baac"
# assert exercise_kth_unique_permutation("aabc", 12) == "cbaa"

## 18. Gợi ý theo mức

**Bài 1**
1. `13 - 1 = 12`. Chia 12 cho 6.
2. `divmod(12, 6)` cho thương và dư nào? `pop(2)` của `['1','2','3','4']` lấy phần tử nào?
3. Thương là **chỉ số** trong `remaining`; sau `pop`, list co lại còn 3 phần tử.

**Bài 2**
1. Bắt đầu với `k = 16`, `remaining = [1,2,3,4]`, `block` lần lượt `6, 2, 1, 1`.
2. Bước 1: `divmod(16, 6)`. Bước 2: chia phần dư cho `2`.
3. Các `idx` thu được chính là các chữ số hệ giai thừa.

**Bài 3**
1. Nhìn lại 5 bước ở đầu mục 9.
2. `range(n, 0, -1)` cho `m = n..1`; `math.factorial(m - 1)` là kích thước nhóm.
3. Đừng quên `k -= 1` trước vòng lặp và `"".join(result)`.

**Bài 4**
1. Đuôi giảm dần không thể tăng thêm → phải sửa số đứng ngay trước đuôi.
2. Hai vòng `while` đi từ phải sang trái: một vòng tìm `a[i] < a[i+1]`, một vòng tìm `a[j] > a[i]`.
3. `a[i], a[j] = a[j], a[i]` rồi `a[i+1:] = reversed(a[i+1:])`; nếu `i < 0` thì `a.reverse()` và `return False`.

**Bài 5**
1. Với `"2314"`: có bao nhiêu hoán vị bắt đầu bằng chữ số **nhỏ hơn** `2`? (`1 × 3! = 6`.)
2. Tổng quát: tại vị trí `i`, gọi `c` = số ký tự còn lại nhỏ hơn `perm[i]`, cộng `c × (số ký tự còn lại sau vị trí i)!`.
3. Tổng các số đó là chỉ số đếm từ 0 → cộng thêm 1.

**Bài 6**
1. Số hoán vị khác nhau của một multiset = `(tổng số ký tự)! / (tích các (số lần lặp)!)`.
2. Dùng `collections.Counter(s)` để giữ số lần còn lại của mỗi ký tự; duyệt `sorted(counter)`.
3. Với mỗi ký tự `ch` có `counter[ch] > 0`: tạm giảm `counter[ch]`, tính số hoán vị phần còn lại; nếu `k <= cnt` thì chốt `ch` và sang vị trí tiếp theo, ngược lại `k -= cnt` và trả `counter[ch]` lại như cũ.

## 19. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1 và Bài 2

**Bài 1:** `divmod(12, 6) = (2, 0)` → in `2 0 3 ['1', '2', '4']`. Chữ số đầu của hoán vị thứ 13 là `3` (khớp bảng nhóm: `k = 13..18` là nhóm "đầu 3").

**Bài 2:** `n = 4, k = 17` → `k = 16`.

| Bước | `remaining` | `block` | `divmod` | Chọn | Kết quả |
|---:|---|---:|---|:---:|---|
| 1 | `[1,2,3,4]` | 6 | `divmod(16, 6) = (2, 4)` | `3` | `3` |
| 2 | `[1,2,4]` | 2 | `divmod(4, 2) = (2, 0)` | `4` | `34` |
| 3 | `[1,2]` | 1 | `divmod(0, 1) = (0, 0)` | `1` | `341` |
| 4 | `[2]` | 1 | `divmod(0, 1) = (0, 0)` | `2` | `3412` |

Hệ giai thừa của `16`: `[2, 2, 0, 0]` (kiểm tra: `2·6 + 2·2 + 0·1 + 0·1 = 16`).

In [31]:
remaining = ['1', '2', '3', '4']
k = 13 - 1
idx, k = divmod(k, 6)
chosen = remaining.pop(idx)
print(idx, k, chosen, remaining)                 # Bài 1

assert perm_factorial(4, 17) == "3412"           # Bài 2
assert to_factoradic(16, 4) == [2, 2, 0, 0]
print("Bài 1, 2: OK")

2 0 3 ['1', '2', '4']
Bài 1, 2: OK


In [32]:
from collections import Counter


def answer_get_permutation(n: int, k: int) -> str:
    remaining = [str(d) for d in range(1, n + 1)]
    k -= 1
    result = []
    for m in range(n, 0, -1):
        idx, k = divmod(k, math.factorial(m - 1))
        result.append(remaining.pop(idx))
    return "".join(result)


def answer_next_permutation(a: list) -> bool:
    i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]:
        i -= 1
    if i < 0:
        a.reverse()
        return False
    j = len(a) - 1
    while a[j] <= a[i]:
        j -= 1
    a[i], a[j] = a[j], a[i]
    a[i + 1:] = reversed(a[i + 1:])
    return True


def answer_permutation_rank(perm: str) -> int:
    remaining = sorted(perm)
    rank0 = 0                                       # chỉ số đếm từ 0
    for i, ch in enumerate(perm):
        smaller = remaining.index(ch)               # số ký tự còn lại nhỏ hơn ch
        rank0 += smaller * math.factorial(len(perm) - 1 - i)
        remaining.pop(smaller)
    return rank0 + 1                                # đổi về đếm từ 1


def count_unique_perms(counter: Counter) -> int:
    total = math.factorial(sum(counter.values()))
    for c in counter.values():
        total //= math.factorial(c)
    return total


def answer_kth_unique_permutation(s: str, k: int) -> str:
    counter = Counter(s)
    result = []
    for _ in range(len(s)):
        for ch in sorted(counter):
            if counter[ch] == 0:
                continue
            counter[ch] -= 1                        # thử đặt ch ở vị trí này
            cnt = count_unique_perms(counter)
            if k <= cnt:                            # đáp án nằm trong nhóm của ch
                result.append(ch)
                break
            k -= cnt                                # bỏ qua cả nhóm của ch
            counter[ch] += 1
    return "".join(result)


# Bài 3
assert answer_get_permutation(3, 3) == "213"
assert answer_get_permutation(4, 9) == "2314"
assert answer_get_permutation(4, 24) == "4321"

# Bài 4
a = [1, 3, 4, 2]; assert answer_next_permutation(a) and a == [1, 4, 2, 3]
a = [3, 2, 1]; assert not answer_next_permutation(a) and a == [1, 2, 3]

# Bài 5: test cụ thể + đối chiếu vét cạn n <= 6 (rank là phép ngược của getPermutation)
assert answer_permutation_rank("2314") == 9
assert answer_permutation_rank("123") == 1
assert answer_permutation_rank("321") == 6
for n in range(1, 7):
    for k in range(1, math.factorial(n) + 1):
        assert answer_permutation_rank(perm_factorial(n, k)) == k

# Bài 6: test cụ thể + đối chiếu với cách liệt kê trên chuỗi ngẫu nhiên nhỏ
assert answer_kth_unique_permutation("aabc", 1) == "aabc"
assert answer_kth_unique_permutation("aabc", 7) == "baac"
assert answer_kth_unique_permutation("aabc", 12) == "cbaa"
random.seed(47)
for _ in range(100):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 6)))
    uniq = sorted(set("".join(p) for p in permutations(s)))
    k = random.randint(1, len(uniq))
    assert answer_kth_unique_permutation(s, k) == uniq[k - 1], (s, k)

print("Tất cả đáp án Bài 3 -> Bài 6: PASS")

Tất cả đáp án Bài 3 -> Bài 6: PASS


## 20. Template ngắn dễ nhớ

```python
def kth_permutation(n, k):
    remaining = [str(d) for d in range(1, n + 1)]   # tăng dần
    k -= 1                                          # (1) đếm từ 0 !!!
    result = []
    for m in range(n, 0, -1):                       # (2) m chữ số còn lại
        idx, k = divmod(k, math.factorial(m - 1))   # (3) nhóm nào + vị trí trong nhóm
        result.append(remaining.pop(idx))           # (4) chốt và xóa
    return "".join(result)
```

Câu thần chú: **"Trừ một — chia nhóm `(m-1)!` — lấy thương chọn số — giữ dư đi tiếp."**

## 21. Checklist trước khi submit

- [ ] Đã `k -= 1` **đúng một lần**, trước vòng lặp?
- [ ] Kích thước nhóm là `(m - 1)!` (không phải `m!`)?
- [ ] Cập nhật `k` bằng **phần dư** (`divmod` trả cả hai)?
- [ ] Dùng `pop(idx)` để không lặp chữ số, và `remaining` ban đầu tăng dần?
- [ ] Trả về **chuỗi** (`"".join` trên list `str`)?
- [ ] Đã thử ca biên: `n = 1`; `k = 1`; `k = n!`; `k` ở cuối/đầu một nhóm (`n=4, k=6` và `k=7`)?
- [ ] Đúng signature `getPermutation(self, n: int, k: int) -> str`?